# 📊 데이터 분석 입문 - 4편: 통계 분석

> **목표**: 상관분석, 그룹 비교(t-test), 교차분석 등 기본 통계를 직접 수행합니다.
>
> **핵심 질문**: "이 차이가 **우연**인가, **의미 있는** 차이인가?"
>
> **소요 시간**: 약 100분

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
!apt-get install -y fonts-nanum > /dev/null 2>&1
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print("✅ 준비 완료!")

In [ ]:
# 분석용 데이터 생성 (ICT 활용과 성취도에 관계가 있도록 설계)
np.random.seed(42)
n = 300

ICT유형 = np.random.choice(['수동적', '능동적'], n, p=[0.45, 0.55])
ICT학교 = np.random.choice([1,2,3,4,5], n, p=[0.1,0.2,0.3,0.25,0.15])
ICT가정 = np.random.choice([1,2,3,4,5], n, p=[0.05,0.15,0.3,0.3,0.2])

# 능동적 활용이 점수에 양의 영향 (현실적 효과 크기)
수학 = (np.random.normal(500, 60, n) 
       + np.where(ICT유형=='능동적', 25, -15)
       + ICT학교 * 4
       + np.random.normal(0, 15, n)).astype(int)

df = pd.DataFrame({
    '학생ID': [f'S{str(i).zfill(3)}' for i in range(1, n+1)],
    '성별': np.random.choice(['남', '여'], n),
    'ICT활용유형': ICT유형,
    'ICT학교사용': ICT학교,
    'ICT가정사용': ICT가정,
    'ICT종합': (ICT학교 + ICT가정) / 2,
    '수학점수': 수학,
    '읽기점수': (수학 + np.random.normal(0, 30, n) - 10).astype(int),
    '과학점수': (수학 + np.random.normal(0, 25, n) + 5).astype(int),
})

print(f"데이터: {df.shape[0]}명, {df.shape[1]}개 변수")
df.head()

---
## 1. 기술통계 (Descriptive Statistics)

분석의 시작은 항상 "데이터가 어떻게 생겼는가?"를 파악하는 것입니다.

In [ ]:
# 전체 기술통계
df.describe().round(1)

In [ ]:
# 그룹별 기술통계 — 핵심!
print("=== ICT 활용 유형별 수학 점수 ===")
그룹별 = df.groupby('ICT활용유형')['수학점수'].agg(['count', 'mean', 'std', 'min', 'max'])
그룹별.columns = ['인원', '평균', '표준편차', '최소', '최대']
print(그룹별.round(1))

**해석 연습:**
- 평균의 차이가 보이나요?
- 하지만 이 차이가 **"우연히"** 생긴 건 아닐까요?
- 이걸 판단하는 게 바로 **통계적 검정**입니다.

---
## 2. 상관분석 (Correlation Analysis)

**"두 변수가 함께 움직이는가?"**를 숫자로 표현합니다.

### 2.1 상관계수 이해하기

**상관계수 (r)**
- **+1에 가까울수록**: 하나가 오르면 다른 것도 오름 (양의 상관)
- **-1에 가까울수록**: 하나가 오르면 다른 것은 내림 (음의 상관)
- **0에 가까울수록**: 관련 없음

| 상관계수 | 해석 |
|---------|------|
| 0.7 ~ 1.0 | 강한 양의 상관 |
| 0.4 ~ 0.7 | 중간 양의 상관 |
| 0.2 ~ 0.4 | 약한 양의 상관 |
| -0.2 ~ 0.2 | 거의 무상관 |
| -0.4 ~ -0.2 | 약한 음의 상관 |

⚠️ **주의: 상관관계 ≠ 인과관계**
"ICT를 많이 쓰면 점수가 높다"가 아니라, "ICT를 많이 쓰는 학생이 점수도 높은 **경향이 있다**"입니다.

In [ ]:
# 상관계수 계산
수치_열 = ['ICT학교사용', 'ICT가정사용', 'ICT종합', '수학점수', '읽기점수', '과학점수']
상관행렬 = df[수치_열].corr().round(3)
print(상관행렬)

In [ ]:
# 히트맵으로 시각화
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(상관행렬, cmap='RdBu_r', vmin=-1, vmax=1)

# 라벨 표시
ax.set_xticks(range(len(수치_열)))
ax.set_yticks(range(len(수치_열)))
ax.set_xticklabels(수치_열, rotation=45, ha='right')
ax.set_yticklabels(수치_열)

# 상관계수 값 표시
for i in range(len(수치_열)):
    for j in range(len(수치_열)):
        ax.text(j, i, f'{상관행렬.iloc[i,j]:.2f}', 
                ha='center', va='center', fontsize=10,
                color='white' if abs(상관행렬.iloc[i,j]) > 0.5 else 'black')

plt.colorbar(im, label='상관계수')
plt.title('변수 간 상관관계 히트맵')
plt.tight_layout()
plt.show()

In [ ]:
# 특정 두 변수의 상관분석 (p-value 포함)
r, p = stats.pearsonr(df['ICT종합'], df['수학점수'])
print(f"ICT종합 vs 수학점수")
print(f"  상관계수 (r): {r:.4f}")
print(f"  p-value: {p:.6f}")
print(f"  통계적으로 유의한가? {'✅ 예 (p < 0.05)' if p < 0.05 else '❌ 아니오 (p >= 0.05)'}")

### p-value란?

**"이 관계가 우연히 생겼을 확률"**이라고 생각하면 됩니다.

- p < 0.05 → 우연일 확률이 5% 미만 → **"통계적으로 유의하다"** (의미 있는 관계)
- p >= 0.05 → 우연일 수 있음 → "유의하지 않다"

⚠️ p-value가 낮다고 해서 관계가 **"강하다"**는 뜻이 아닙니다!
상관계수(r)의 크기가 관계의 강도, p-value는 관계의 신뢰도입니다.

### ✏️ 연습 4-1
`ICT가정사용`과 `읽기점수`의 상관계수와 p-value를 구하고, 산점도를 그려보세요.

In [ ]:
# ✏️ 여기에 코드를 작성하세요

# 상관계수 계산
r, p = stats.pearsonr(___, ___)
print(f"상관계수: {r:.4f}, p-value: {p:.6f}")

# 산점도
plt.figure(figsize=(8, 6))
plt.scatter(___, ___, alpha=0.4)
plt.xlabel(___)
plt.ylabel(___)
plt.title(f'상관계수 r = {r:.3f}, p = {p:.4f}')
plt.show()

---
## 3. 단순 선형회귀 (Simple Linear Regression)

상관분석은 "두 변수가 함께 움직이는가?"까지만 답합니다.
**회귀분석**은 한 걸음 더 나아가 **"X로 Y를 예측하면 어떻게 되는가?"**에 답합니다.

직선의 방정식을 기억하나요? **`Y = a × X + b`**
- **a (기울기)**: X가 1 증가할 때 Y가 평균 얼마나 변하는가
- **b (절편)**: X가 0일 때 Y의 값
- 회귀분석은 데이터에 **가장 잘 맞는 직선**(실제값과 예측값의 차이 제곱합이 최소인 직선)을 찾아줍니다.

In [ ]:
# ICT종합(X)으로 수학점수(Y)를 예측하는 직선 찾기
# scipy의 linregress 하나면 됩니다 (맨 위에서 이미 import한 stats 사용)
x = df['ICT종합']
y = df['수학점수']

결과 = stats.linregress(x, y)

print(f"기울기 (a): {결과.slope:.2f}")
print(f"절편 (b): {결과.intercept:.2f}")
print(f"결정계수 (R²): {결과.rvalue**2:.3f}")
print(f"p-value: {결과.pvalue:.6f}")
print()
print(f"📐 회귀식:  수학점수 = {결과.slope:.1f} × ICT종합 + {결과.intercept:.1f}")

**해석하는 법:**
- **기울기**: ICT 종합 사용이 1점 오를 때 수학 점수가 평균 몇 점 변하는지.
- **R² (결정계수)**: 수학 점수의 흩어짐(분산) 중 ICT로 **설명되는 비율**. 0~1 사이이고 1에 가까울수록 직선이 데이터를 잘 설명합니다.
  - 예) R²=0.04 → ICT가 점수 차이의 **4%만** 설명. 나머지 96%는 다른 요인.
- **p-value**: 이 기울기가 실제로는 0인데 우연히 나왔을 확률 (0.05 미만이면 유의).

⚠️ **R²가 낮아도 정상입니다.** 사람의 성취도는 수십 가지 요인이 얽혀 있어서, 변수 하나로 설명되는 비율은 원래 작습니다. "낮은 R² = 분석 실패"가 아닙니다.

In [ ]:
# 회귀선 그리기 + 예측해보기
plt.figure(figsize=(8, 6))
plt.scatter(x, y, alpha=0.4, color='steelblue', label='실제 데이터')

# 회귀선
x_line = np.linspace(x.min(), x.max(), 100)
y_line = 결과.slope * x_line + 결과.intercept
plt.plot(x_line, y_line, 'r-', linewidth=2, label='회귀선 (예측)')

plt.xlabel('ICT 종합 사용')
plt.ylabel('수학 점수')
plt.title(f'회귀식: y = {결과.slope:.1f}x + {결과.intercept:.1f}   (R² = {결과.rvalue**2:.3f})')
plt.legend()
plt.show()

# 예측: ICT 종합이 4점인 학생의 예상 수학 점수는?
ict_값 = 4
예측점수 = 결과.slope * ict_값 + 결과.intercept
print(f"💡 ICT 종합 {ict_값}점인 학생의 예상 수학 점수: {예측점수:.1f}점")

### ✏️ 연습 4-3
`ICT학교사용`(X)으로 `읽기점수`(Y)를 예측하는 단순 선형회귀를 실행하세요.
기울기·R²를 출력하고, **ICT학교사용이 5점인 학생의 읽기점수**를 예측해보세요.

In [ ]:
# ✏️ 여기에 코드를 작성하세요
결과 = stats.linregress(___, ___)
print(f"기울기: {결과.slope:.2f}, R²: {결과.rvalue**2:.3f}")

예측 = ___   # 힌트: 결과.slope * 5 + 결과.intercept
print(f"ICT학교사용 5점 → 읽기점수 예측: {예측:.1f}점")

---
## 4. t-검정 (t-test) — 두 그룹의 평균 비교

**"능동적 활용 그룹과 수동적 활용 그룹의 수학 점수 평균이 정말 다른가?"**

In [ ]:
# 두 그룹 분리
능동적 = df[df['ICT활용유형'] == '능동적']['수학점수']
수동적 = df[df['ICT활용유형'] == '수동적']['수학점수']

print(f"능동적 활용 ({len(능동적)}명): 평균 {능동적.mean():.1f}, 표준편차 {능동적.std():.1f}")
print(f"수동적 활용 ({len(수동적)}명): 평균 {수동적.mean():.1f}, 표준편차 {수동적.std():.1f}")
print(f"평균 차이: {능동적.mean() - 수동적.mean():.1f}점")

In [ ]:
# 독립 표본 t-검정 (independent samples t-test)
t통계량, p값 = stats.ttest_ind(능동적, 수동적)

print(f"\n=== t-검정 결과 ===")
print(f"t 통계량: {t통계량:.4f}")
print(f"p-value: {p값:.6f}")
print()
if p값 < 0.05:
    print("✅ 두 그룹의 수학 점수 평균은 통계적으로 유의한 차이가 있습니다.")
    print(f"   (능동적 활용 그룹이 {능동적.mean() - 수동적.mean():.1f}점 높음)")
else:
    print("❌ 두 그룹의 수학 점수 평균 차이는 통계적으로 유의하지 않습니다.")

In [ ]:
# 시각화: 두 그룹 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 히스토그램 겹쳐 그리기
axes[0].hist(능동적, bins=20, alpha=0.6, label=f'능동적 (평균:{능동적.mean():.0f})', color='#2ecc71')
axes[0].hist(수동적, bins=20, alpha=0.6, label=f'수동적 (평균:{수동적.mean():.0f})', color='#e74c3c')
axes[0].axvline(능동적.mean(), color='#27ae60', linestyle='--', linewidth=2)
axes[0].axvline(수동적.mean(), color='#c0392b', linestyle='--', linewidth=2)
axes[0].set_xlabel('수학 점수')
axes[0].set_ylabel('학생 수')
axes[0].set_title('ICT 활용 유형별 점수 분포')
axes[0].legend()

# 박스플롯
데이터 = [수동적, 능동적]
bp = axes[1].boxplot(데이터, tick_labels=['수동적', '능동적'], patch_artist=True)
bp['boxes'][0].set_facecolor('#ffcccc')
bp['boxes'][1].set_facecolor('#ccffcc')
axes[1].set_ylabel('수학 점수')
axes[1].set_title(f't-test: t={t통계량:.2f}, p={p값:.4f}')

plt.tight_layout()
plt.show()

### ✏️ 연습 4-2
성별(남/여)에 따른 수학 점수 평균에 유의한 차이가 있는지 t-검정을 실시하세요.

In [ ]:
# ✏️ 여기에 코드를 작성하세요
남 = df[df['성별'] == '남']['수학점수']
여 = df[df['성별'] == '여']['수학점수']

print(f"남학생 평균: {남.mean():.1f}")
print(f"여학생 평균: {여.mean():.1f}")

t, p = stats.ttest_ind(___, ___)
print(f"\nt = {t:.4f}, p = {p:.6f}")
print("유의한 차이가 있는가?", ___)

---
## 5. 일원분산분석 (One-way ANOVA) — 3개 이상 그룹 비교

t-test는 2개 그룹만 비교할 수 있습니다.
3개 이상 그룹을 비교하려면 **ANOVA**를 씁니다.

In [ ]:
# ICT 사용 수준 3그룹 만들기
df['ICT수준'] = pd.cut(df['ICT종합'], bins=[0, 2, 3.5, 5],
                      labels=['저사용', '중사용', '고사용'])

# 그룹별 기술통계
print(df.groupby('ICT수준')['수학점수'].agg(['count', 'mean', 'std']).round(1))

In [ ]:
# ANOVA 실시
저 = df[df['ICT수준'] == '저사용']['수학점수']
중 = df[df['ICT수준'] == '중사용']['수학점수']
고 = df[df['ICT수준'] == '고사용']['수학점수']

F통계량, p값 = stats.f_oneway(저, 중, 고)
print(f"=== ANOVA 결과 ===")
print(f"F 통계량: {F통계량:.4f}")
print(f"p-value: {p값:.6f}")
if p값 < 0.05:
    print("\n✅ 세 그룹 중 적어도 하나의 평균이 유의하게 다릅니다.")
else:
    print("\n❌ 세 그룹의 평균에 유의한 차이가 없습니다.")

---
## 6. 교차분석 (Cross-tabulation) — 범주 × 범주

**"ICT 활용 유형과 성별 사이에 관련이 있는가?"** 같은 질문.

In [ ]:
# 교차표 만들기
교차표 = pd.crosstab(df['ICT활용유형'], df['성별'], margins=True, margins_name='합계')
print(교차표)

In [ ]:
# 비율로 보기 (더 해석하기 쉬움)
비율표 = pd.crosstab(df['ICT활용유형'], df['성별'], normalize='index') * 100
print(비율표.round(1))

In [ ]:
# 카이제곱 검정 — 두 범주형 변수 간 관련성 검정
교차표_순수 = pd.crosstab(df['ICT활용유형'], df['성별'])
chi2, p, dof, expected = stats.chi2_contingency(교차표_순수)

print(f"=== 카이제곱 검정 결과 ===")
print(f"카이제곱 통계량: {chi2:.4f}")
print(f"p-value: {p:.4f}")
print(f"자유도: {dof}")
if p < 0.05:
    print("\n✅ ICT 활용 유형과 성별 사이에 유의한 관련이 있습니다.")
else:
    print("\n❌ 두 변수 사이에 유의한 관련이 없습니다.")

---
## 7. 분석 결과를 문장으로 쓰기

통계 분석의 마지막 단계는 **숫자를 해석 가능한 문장으로** 바꾸는 것입니다.
보고서에 들어갈 형식으로 연습해봅시다.

In [ ]:
# 분석 결과 자동 요약 함수
def 분석_요약(df):
    print("=" * 60)
    print("📋 분석 결과 요약")
    print("=" * 60)
    
    # 1. 기술통계
    print(f"\n1. 기본 정보")
    print(f"   - 분석 대상: {len(df)}명")
    print(f"   - 수학 평균: {df['수학점수'].mean():.1f}점 (SD={df['수학점수'].std():.1f})")
    
    # 2. 그룹 비교
    능동 = df[df['ICT활용유형']=='능동적']['수학점수']
    수동 = df[df['ICT활용유형']=='수동적']['수학점수']
    t, p = stats.ttest_ind(능동, 수동)
    
    print(f"\n2. ICT 활용 유형별 수학 점수 비교")
    print(f"   - 능동적 활용: M={능동.mean():.1f}, SD={능동.std():.1f} (n={len(능동)})")
    print(f"   - 수동적 활용: M={수동.mean():.1f}, SD={수동.std():.1f} (n={len(수동)})")
    print(f"   - t({len(능동)+len(수동)-2})={t:.2f}, p={'<.001' if p<0.001 else f'={p:.3f}'}")
    유의 = "유의한 차이가 있었다" if p < 0.05 else "유의한 차이가 없었다"
    print(f"   → 두 그룹 간 {유의}.")
    
    # 3. 상관분석
    r, p_corr = stats.pearsonr(df['ICT종합'], df['수학점수'])
    print(f"\n3. ICT 종합 사용과 수학 점수의 상관")
    print(f"   - r={r:.3f}, p={'<.001' if p_corr<0.001 else f'={p_corr:.3f}'}")
    강도 = "강한" if abs(r)>0.5 else "중간" if abs(r)>0.3 else "약한"
    방향 = "양의" if r > 0 else "음의"
    print(f"   → {강도} {방향} 상관이 나타났다.")
    
    print("\n" + "=" * 60)

분석_요약(df)

---
## 📝 4편 정리 — 통계 검정 선택 가이드

| 질문 | 방법 | 코드 |
|------|------|------|
| 두 수치 변수의 관계? | 상관분석 | `stats.pearsonr(x, y)` |
| 한 변수로 다른 변수 예측? | 단순 선형회귀 | `stats.linregress(x, y)` |
| 두 그룹의 평균 차이? | t-검정 | `stats.ttest_ind(a, b)` |
| 세 그룹 이상의 평균? | ANOVA | `stats.f_oneway(a, b, c)` |
| 범주 × 범주 관련성? | 카이제곱 | `stats.chi2_contingency()` |

**모든 검정의 핵심:**
- p < 0.05 → 통계적으로 유의함 (우연이 아닐 가능성 95% 이상)
- 효과 크기(r, 평균 차이)도 함께 보고해야 의미 있음

### 다음 편 예고
5편에서는 PISA 실제 데이터로 **종합 프로젝트**를 수행합니다!